In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [7]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# ====== EDIT THIS PATH to match your Kaggle dataset folder ======
TRAIN_PATH = "/kaggle/input/competitions/comment-category-prediction-challenge/train.csv"  # <-- change folder name
# ===============================================================

df = pd.read_csv(TRAIN_PATH)

TARGET = "label"
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

a, b = X_train.shape
c, d = X_val.shape

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("Q1 answer (a + c):", a + c)

X_train shape: (158400, 14)
X_val shape: (39600, 14)
Q1 answer (a + c): 198000


In [8]:
X_train_dt = X_train.copy()
X_val_dt   = X_val.copy()

# Parse created_date
X_train_dt["created_date"] = pd.to_datetime(X_train_dt["created_date"], errors="coerce")
X_val_dt["created_date"]   = pd.to_datetime(X_val_dt["created_date"], errors="coerce")

# Extract features
for D in (X_train_dt, X_val_dt):
    D["day"] = D["created_date"].dt.day
    D["month"] = D["created_date"].dt.month
    D["year"] = D["created_date"].dt.year

# "most frequently occurring month across all years in X_train"
most_common_month = X_train_dt["month"].value_counts(dropna=True).idxmax()
print("Q2 most frequent month in X_train:", most_common_month)
print(X_train_dt["month"].value_counts(dropna=True).head(12))

Q2 most frequent month in X_train: 5
month
5     18847
3     17072
4     16126
2     15571
1     15200
12    12293
8     11340
10    11087
11    10805
6     10691
9      9966
7      9402
Name: count, dtype: int64


In [9]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

X_train_3 = X_train.copy()
X_val_3   = X_val.copy()

# categorical columns to impute with 'none' (object/category)
cat_cols = X_train_3.select_dtypes(include=["object", "category"]).columns.tolist()

# Impute categorical missing with 'none'
cat_imputer = SimpleImputer(strategy="constant", fill_value="none")
X_train_3[cat_cols] = cat_imputer.fit_transform(X_train_3[cat_cols])
X_val_3[cat_cols]   = cat_imputer.transform(X_val_3[cat_cols])

# Only one-hot encode these three columns (as requested)
ohe_cols = ["religion", "gender", "race"]
missing_ohe = [c for c in ohe_cols if c not in X_train_3.columns]
if missing_ohe:
    raise ValueError(f"These OHE columns are missing from data: {missing_ohe}")

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe_train = ohe.fit_transform(X_train_3[ohe_cols])
ohe_val   = ohe.transform(X_val_3[ohe_cols])

ohe_feature_names = ohe.get_feature_names_out(ohe_cols)

# Drop original cols and add OHE columns back, keep as DataFrame
X_train_3_out = X_train_3.drop(columns=ohe_cols).copy()
X_val_3_out   = X_val_3.drop(columns=ohe_cols).copy()

X_train_3_out[ohe_feature_names] = ohe_train
X_val_3_out[ohe_feature_names]   = ohe_val

print("X_train after imputing + OHE(selected) shape:", X_train_3_out.shape)
print("Q3 answer (b = #features):", X_train_3_out.shape[1])

X_train after imputing + OHE(selected) shape: (158400, 30)
Q3 answer (b = #features): 30


In [10]:
from sklearn.feature_extraction.text import CountVectorizer

# Use the output from step 3 (X_train_3_out / X_val_3_out)
if "comment" not in X_train_3_out.columns:
    raise ValueError("Column 'comment' not found (needed for CountVectorizer).")

# Ensure string and handle missing safely
train_comments = X_train_3_out["comment"].fillna("").astype(str)
val_comments   = X_val_3_out["comment"].fillna("").astype(str)

cv = CountVectorizer()
Xtr_counts = cv.fit_transform(train_comments)
Xva_counts = cv.transform(val_comments)

# "document at index 1" means second row of X_train
row1_sum = int(Xtr_counts[1].sum())
print("Q4 answer (sum of counts in X_train doc index 1):", row1_sum)

Q4 answer (sum of counts in X_train doc index 1): 41


In [11]:
def map_disability(s: pd.Series) -> pd.Series:
    # Normalize to string lowercase, but keep NaN
    x = s.copy()
    # common cases: bool, 0/1, "True"/"False"
    # convert to string for robust mapping, but preserve NaNs
    mask_na = x.isna()
    x = x.astype(str).str.strip().str.lower()
    x[mask_na] = np.nan

    mapping = {
        "true": 1, "false": 0,
        "1": 1, "0": 0,
        "yes": 1, "no": 0,
        "t": 1, "f": 0
    }
    out = x.map(mapping)

    # if already numeric but got unmapped strings, try numeric coercion
    out = out.fillna(pd.to_numeric(x, errors="coerce"))

    return out

X_train_5 = X_train.copy()
X_val_5   = X_val.copy()

if "disability" not in X_train_5.columns:
    raise ValueError("Column 'disability' not found.")

X_train_5["disability"] = map_disability(X_train_5["disability"]).astype("Int64")
X_val_5["disability"]   = map_disability(X_val_5["disability"]).astype("Int64")

# If there are still NaNs, decide how to treat; question implies after transformation
# We'll treat missing as 0 for the sum, but print missing count so you know.
train_missing = int(X_train_5["disability"].isna().sum())
val_missing = int(X_val_5["disability"].isna().sum())
print("Missing disability values (train, val):", train_missing, val_missing)

total_sum = int(X_train_5["disability"].fillna(0).sum() + X_val_5["disability"].fillna(0).sum())
print("Q5 answer (sum of disability over train+val):", total_sum)

Missing disability values (train, val): 0 0
Q5 answer (sum of disability over train+val): 2743


In [12]:
from sklearn.preprocessing import StandardScaler

X_train_6 = X_train.copy()
X_val_6   = X_val.copy()

# Convert created_date to datetime then drop (so no datetime columns remain)
X_train_6["created_date"] = pd.to_datetime(X_train_6["created_date"], errors="coerce")
X_val_6["created_date"]   = pd.to_datetime(X_val_6["created_date"], errors="coerce")

# Drop datetime columns
dt_cols_train = X_train_6.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()
X_train_6 = X_train_6.drop(columns=dt_cols_train)
X_val_6   = X_val_6.drop(columns=dt_cols_train)

# Select numeric columns to scale
num_cols = X_train_6.select_dtypes(include=[np.number]).columns.tolist()

scaler = StandardScaler()
scaler.fit(X_train_6[num_cols])

print("Numeric columns scaled:", num_cols)
print("Q6 answer (# features seen during fit):", scaler.n_features_in_)

Numeric columns scaled: ['post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote', 'if_1', 'if_2']
Q6 answer (# features seen during fit): 8


In [17]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score

X_train7, X_val7, y_train7, y_val7 = train_test_split(
    X, y, test_size=0.2, random_state=42
)

TEXT_COL = "comment"
DATE_COL = "created_date"

def add_date_parts(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    dt = pd.to_datetime(df[DATE_COL], errors="coerce")
    df["day"] = dt.dt.day
    df["month"] = dt.dt.month
    df["year"] = dt.dt.year
    return df.drop(columns=[DATE_COL])

date_fe = FunctionTransformer(add_date_parts, validate=False)

# Fit once to infer types after date FE
Xtr_fe = date_fe.fit_transform(X_train7)

num_cols = Xtr_fe.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = Xtr_fe.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# Remove text col from cat_cols
if TEXT_COL in num_cols: num_cols.remove(TEXT_COL)
if TEXT_COL in cat_cols: cat_cols.remove(TEXT_COL)

# ---- Transformers ----
def clean_text(x):
    # x may be a Series or a (n,1) array; convert to 1D iterable of strings
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]
    if isinstance(x, pd.Series):
        return x.fillna("").astype(str).to_numpy()
    x = np.asarray(x)
    if x.ndim == 2 and x.shape[1] == 1:
        x = x[:, 0]
    return pd.Series(x).fillna("").astype(str).to_numpy()

text_pipe = Pipeline(steps=[
    ("to_str", FunctionTransformer(clean_text, validate=False)),
    ("tfidf", TfidfVectorizer(stop_words="english")),
])

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("abs", FunctionTransformer(np.abs, validate=False)),
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_pipe, TEXT_COL),
        ("cat", cat_pipe, cat_cols),
        ("num", num_pipe, num_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

pipe = Pipeline(steps=[
    ("date_parts", date_fe),
    ("prep", preprocessor),
    ("clf", MultinomialNB()),
])
pipe.fit(X_train7, y_train7)

pred_train = pipe.predict(X_train7)
pred_val   = pipe.predict(X_val7)

f1_train = f1_score(y_train7, pred_train, average="macro")
f1_val   = f1_score(y_val7, pred_val, average="macro")

print("Q7 macro F1 on TRAIN:", f1_train)
print("Q8 macro F1 on VAL:", f1_val)

Q7 macro F1 on TRAIN: 0.48157528607647226
Q8 macro F1 on VAL: 0.45818801404562537


In [19]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score

X_train8, X_val8, y_train8, y_val8 = train_test_split(
    X, y, test_size=0.2, random_state=42
)

TEXT_COL = "comment"
DATE_COL = "created_date"

def add_date_parts_and_weekend(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    dt = pd.to_datetime(df[DATE_COL], errors="coerce")
    df["day"] = dt.dt.day
    df["month"] = dt.dt.month
    df["year"] = dt.dt.year
    df["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)  # 1/0
    return df.drop(columns=[DATE_COL])

def clean_text(x):
    # Return 1D array of strings for TfidfVectorizer
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]
    if isinstance(x, pd.Series):
        return x.fillna("").astype(str).to_numpy()
    x = np.asarray(x)
    if x.ndim == 2 and x.shape[1] == 1:
        x = x[:, 0]
    return pd.Series(x).fillna("").astype(str).to_numpy()

date_fe2 = FunctionTransformer(add_date_parts_and_weekend, validate=False)

# infer types after FE
Xtr_fe2 = date_fe2.fit_transform(X_train8)

num_cols2 = Xtr_fe2.select_dtypes(include=[np.number]).columns.tolist()
cat_cols2 = Xtr_fe2.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# Treat is_weekend as categorical as requested:
# move from numeric -> categorical by converting it to string during transform
# easiest: remove from numeric and include it in cat as column, but we must convert it before OHE
# We'll instead keep it numeric and ALSO one-hot it by casting in a small transformer within cat pipe.
# Simpler: just move it to cat and cast column to string in a transformer applied to cat subset.

if "is_weekend" in num_cols2:
    num_cols2.remove("is_weekend")
    cat_cols2.append("is_weekend")

# Remove text col from cat/num lists
if TEXT_COL in num_cols2: num_cols2.remove(TEXT_COL)
if TEXT_COL in cat_cols2: cat_cols2.remove(TEXT_COL)

def cat_to_str(df_cat):
    # df_cat is a DataFrame subset; convert all to string to be safe for OHE
    if isinstance(df_cat, pd.DataFrame):
        return df_cat.astype(str)
    return pd.DataFrame(df_cat).astype(str)

text_pipe2 = Pipeline(steps=[
    ("to_str", FunctionTransformer(clean_text, validate=False)),
    ("tfidf", TfidfVectorizer(stop_words="english")),
])

num_pipe2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("abs", FunctionTransformer(np.abs, validate=False)),  # abs only on numeric
])

cat_pipe2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_str", FunctionTransformer(cat_to_str, validate=False)),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor2 = ColumnTransformer(
    transformers=[
        ("text", text_pipe2, TEXT_COL),
        ("cat", cat_pipe2, cat_cols2),
        ("num", num_pipe2, num_cols2),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

pipe2 = Pipeline(steps=[
    ("date_parts", date_fe2),
    ("prep", preprocessor2),
    ("clf", MultinomialNB()),
])

pipe2.fit(X_train8, y_train8)

pred_train2 = pipe2.predict(X_train8)
pred_val2   = pipe2.predict(X_val8)

f1_train2 = f1_score(y_train8, pred_train2, average="macro")
f1_val2   = f1_score(y_val8, pred_val2, average="macro")

print("Q9 macro F1 on TRAIN (rounded 4):", round(f1_train2, 4))
print("Q10 macro F1 on VAL (rounded 4):", round(f1_val2, 4))

Q9 macro F1 on TRAIN (rounded 4): 0.4815
Q10 macro F1 on VAL (rounded 4): 0.4581


In [20]:
# After you have `pipe` (Q7 model) and `pipe2` (Q9 model) fitted:

Xt7 = pipe.named_steps["prep"].transform(pipe.named_steps["date_parts"].transform(X_train7))
Xt9 = pipe2.named_steps["prep"].transform(pipe2.named_steps["date_parts"].transform(X_train8))

print("Q7 transformed feature matrix shape:", Xt7.shape)
print("Q9 transformed feature matrix shape:", Xt9.shape)
print("Extra features in Q9 vs Q7:", Xt9.shape[1] - Xt7.shape[1])

Q7 transformed feature matrix shape: (158400, 149415)
Q9 transformed feature matrix shape: (158400, 149417)
Extra features in Q9 vs Q7: 2


In [21]:
tmp = X_train8.copy()
dt = pd.to_datetime(tmp["created_date"], errors="coerce")
is_weekend = dt.dt.dayofweek.isin([5,6])
print("Non-null created_date:", dt.notna().sum(), "out of", len(dt))
print("Weekend counts (excluding NaT):")
print(is_weekend.value_counts(dropna=True))

Non-null created_date: 158400 out of 158400
Weekend counts (excluding NaT):
created_date
False    108746
True      49654
Name: count, dtype: int64


In [22]:
Xt7 = pipe.named_steps["prep"].transform(pipe.named_steps["date_parts"].transform(X_train7))
Xt9 = pipe2.named_steps["prep"].transform(pipe2.named_steps["date_parts"].transform(X_train8))

print("Q7 transformed shape:", Xt7.shape)
print("Q9 transformed shape:", Xt9.shape)
print("Extra features in Q9:", Xt9.shape[1] - Xt7.shape[1])

Q7 transformed shape: (158400, 149415)
Q9 transformed shape: (158400, 149417)
Extra features in Q9: 2


In [23]:
pred_train_q7 = pipe.predict(X_train7)
pred_val_q7   = pipe.predict(X_val7)

pred_train_q9 = pipe2.predict(X_train8)
pred_val_q9   = pipe2.predict(X_val8)

print("Train predictions identical?", np.array_equal(pred_train_q7, pred_train_q9))
print("Val predictions identical?", np.array_equal(pred_val_q7, pred_val_q9))

print("Train #different:", np.sum(pred_train_q7 != pred_train_q9))
print("Val   #different:", np.sum(pred_val_q7 != pred_val_q9))

Train predictions identical? False
Val predictions identical? False
Train #different: 27
Val   #different: 7


In [24]:
from sklearn.metrics import f1_score

f1_train_q7 = f1_score(y_train7, pipe.predict(X_train7), average="macro")
f1_val_q7   = f1_score(y_val7, pipe.predict(X_val7), average="macro")

f1_train_q9 = f1_score(y_train8, pipe2.predict(X_train8), average="macro")
f1_val_q9   = f1_score(y_val8, pipe2.predict(X_val8), average="macro")

print("Train F1 Q7:", format(f1_train_q7, ".10f"))
print("Train F1 Q9:", format(f1_train_q9, ".10f"))
print("Val   F1 Q7:", format(f1_val_q7, ".10f"))
print("Val   F1 Q9:", format(f1_val_q9, ".10f"))
print("Train delta:", f1_train_q9 - f1_train_q7)
print("Val   delta:", f1_val_q9 - f1_val_q7)

Train F1 Q7: 0.4815752861
Train F1 Q9: 0.4815170592
Val   F1 Q7: 0.4581880140
Val   F1 Q9: 0.4581156825
Train delta: -5.8226871081257237e-05
Val   delta: -7.233155993535689e-05
